In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf


from scipy.stats import permutation_test



In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_path         → g:\MOUS_204\MOUS_visual\output_source\source_block

In [3]:
# acw_all = pd.read_pickle(ACW_path /f"autocorrelation_subjects_all.pickle")

#lectura de acw_50
#lectura de acw_50
acw_50_df = pd.read_pickle(ACW_path /f"acw_50_df.pickle")




##creacion dataframe acw_50

# acw_50_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_50_elect_all_epoch_all"]]
# acw_50_df.to_pickle(ACW_path / f"acw_50_df.pickle")

# acw_0_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_0_elect_all_epoch_all"]]
# acw_0_df.to_pickle(ACW_path / f"acw_0_df.pickle")

# del acw_all, del acw_0_df

In [4]:
def print5(*args):
    print(*(f"{x:.5f}" if isinstance(x, float) else x for x in args))

channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

channels_mag=channels_mag.tolist()
print5(channels_mag)
indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels






##valores de las columnas
condition = acw_50_df["Condition"].unique()
print("Condiciones en los datos:", condition)

subjects = acw_50_df["Subject"].unique()
print("Sujetos en los datos:", subjects)

elect_all= acw_50_df["Elect"].unique()
print("sensores en los datos:", elect_all)

epochs_all= acw_50_df["Epoch"].unique()
print("Epochs en los datos:", epochs_all)
## get info 

#read epochs to build evoked
epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{subj}_epochs_zinnen_block-epo.fif")
len(epochs_zinnen.pick("mag", exclude="bads").ch_names)

evokeds_zinnen=epochs_zinnen.average()
## evokeds_zinnen es una lista

##cojo el primer elemento, solo tengo una lista
evoked_zinnen=evokeds_zinnen

info=evoked_zinnen.info
del epochs_zinnen
del evokeds_zinnen
del evoked_zinnen




['MLC11-4304', 'MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', 'MLO4

## Selection of values

In [5]:
##selección de valores
##matrices de valores de acw_50, divididas por condicion y tipo de channel
X_zinnen = []

X_woorden = []


##Filtras la condicion
for cond in condition:
    acw_50_condition_df=acw_50_df[acw_50_df["Condition"] == f"{cond}"]


#         #filtras por sujeto 
    for subj in subjects:
    #creacion de lista de valores de acw_50 para cada sujeto
        acw_50_epoch_list= []
#       #filtras por sujeto
        for epoch in epochs_all:
            # Extraer el valor de acw_50 para la combinación actual
            try:
            #coges el valor de acw_50 para el sujeto y el epoch
                print(f"subj:{subj},epoch, {epoch}")
                acw_50_elect_all_epoch_all = acw_50_condition_df[(acw_50_condition_df["Subject"] == subj) & (acw_50_condition_df["Epoch"] == epoch)]["acw_50_elect_all_epoch_all"]
                if not acw_50_elect_all_epoch_all.empty:
                    acw_50_epoch_list.append(acw_50_elect_all_epoch_all)

            except Exception as e:
                print(e, "probablemente faltaban epochs en algunos sujetos")
                continue
        # Convertir la lista a un array de numpy
        acw_50_epoch_array = np.array(acw_50_epoch_list)
        #take the mean on epochs
        acw_50_epoch_mean = np.mean(acw_50_epoch_array, axis=0)

        # #add it to the different lists
        if cond == "zinnen":
            X_zinnen.append(acw_50_epoch_mean)

        if cond == "woorden":
            X_woorden.append(acw_50_epoch_mean)


X_zinnen = np.array(X_zinnen)
X_woorden = np.array(X_woorden)

##take the mean on channels
X_zinnen_subj_mean = np.mean(X_zinnen, axis=1)
X_woorden_subj_mean = np.mean(X_woorden, axis=1)



print("Shapes de las matrices de valores acw_50:\n")

print("▶ Condición: ZINNEN")
print("Condition zinnen:", X_zinnen_subj_mean.shape)
print("Condition woorden:", X_woorden_subj_mean.shape)


##take the mean on channels
X_zinnen_elect_mean = np.mean(X_zinnen, axis=0)
X_woorden_elect_mean = np.mean(X_woorden, axis=0)



print("Shapes de las matrices de valores acw_50:\n")

print("▶ Condición: ZINNEN")
print("Condition zinnen elect:", X_zinnen_elect_mean.shape)
print("Condition woorden elect:", X_woorden_elect_mean.shape)

subj:sub-V1001,epoch, 0
subj:sub-V1001,epoch, 1
subj:sub-V1001,epoch, 2
subj:sub-V1001,epoch, 3
subj:sub-V1001,epoch, 4
subj:sub-V1001,epoch, 5
subj:sub-V1001,epoch, 6
subj:sub-V1001,epoch, 7
subj:sub-V1001,epoch, 8
subj:sub-V1001,epoch, 9
subj:sub-V1001,epoch, 10
subj:sub-V1001,epoch, 11
subj:sub-V1001,epoch, 12
subj:sub-V1001,epoch, 13
subj:sub-V1001,epoch, 14
subj:sub-V1001,epoch, 15
subj:sub-V1001,epoch, 16
subj:sub-V1001,epoch, 17
subj:sub-V1001,epoch, 18
subj:sub-V1001,epoch, 19
subj:sub-V1001,epoch, 20
subj:sub-V1001,epoch, 21
subj:sub-V1001,epoch, 22
subj:sub-V1001,epoch, 23
subj:sub-V1002,epoch, 0
subj:sub-V1002,epoch, 1
subj:sub-V1002,epoch, 2
subj:sub-V1002,epoch, 3
subj:sub-V1002,epoch, 4
subj:sub-V1002,epoch, 5
subj:sub-V1002,epoch, 6
subj:sub-V1002,epoch, 7
subj:sub-V1002,epoch, 8
subj:sub-V1002,epoch, 9
subj:sub-V1002,epoch, 10
subj:sub-V1002,epoch, 11
subj:sub-V1002,epoch, 12
subj:sub-V1002,epoch, 13
subj:sub-V1002,epoch, 14
subj:sub-V1002,epoch, 15
subj:sub-V1002,epoch

# plots for general values

### plot_topomap

In [6]:
# for plot_topomap I use the mean of epochs and subjects so the structure for each condition is (channels,)
acw_50_zinnen = np.mean(X_zinnen, axis=0)
acw_50_woorden = np.mean(X_woorden, axis=0)

import matplotlib.pyplot as plt

for cond in condition:
    if cond == "zinnen":
        exp_condition = "zinnen"
        values = acw_50_zinnen
    elif cond == "woorden":
        exp_condition = "woorden"
        values = acw_50_woorden

    # Crea figura + eje manual
    fig, ax = plt.subplots(figsize=(6, 6))

    # Dibujar topomap DENTRO de ese eje
    im, _ = mne.viz.plot_topomap(
        data=values,
        pos=info,
        axes=ax,           # ← esto asegura que se dibuja donde tú quieres
        cmap='RdBu_r',
        vlim=(-np.max(np.abs(values)), np.max(np.abs(values))),
        mask=None,
        contours=0,
        show=False         # ← evita que se muestre automáticamente
    )

    # Título
    fig.suptitle(f"acw_50 values in {exp_condition}", fontsize=20)

    # Añadir barra de color (leyenda)
    cbar = fig.colorbar(im, ax=ax, shrink=0.6)
    cbar.set_label("acw_50 value")

    # Mostrar figura
    plt.show()


### linear plots

In [7]:
# Import KDE function from scipy
from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt

# Make a copy of the original data (not strictly necessary unless modifying them later)
X_zinnen_copy = X_zinnen.copy()
X_woorden_copy = X_woorden.copy()

# Flatten arrays in case they have more than 1 dimension (e.g., shape (n,1))
X_zinnen_flat = np.array(X_zinnen).flatten()
X_woorden_flat = np.array(X_woorden).flatten()

# Create a new figure for the histogram
plt.figure(figsize=(10, 5))

# Plot histogram for Zinnen condition
# bins=60 gives higher resolution; alpha=0.4 makes bars transparent
counts_zinnen, bins_zinnen, _ = plt.hist(
    X_zinnen_flat, bins=150, alpha=0.4, label='Zinnen', color='blue'
)

# Plot histogram for Woorden condition
counts_woorden, bins_woorden, _ = plt.hist(
    X_woorden_flat, bins=150, alpha=0.4, label='Woorden', color='red'
)

# Create a KDE (Kernel Density Estimation) for Zinnen
kde_zinnen = gaussian_kde(X_zinnen_flat)

# Create x-values for the KDE curve
x_vals_zinnen = np.linspace(X_zinnen_flat.min(), X_zinnen_flat.max(), 300)

# Scale the KDE to match histogram counts
# This is important so the curve height aligns with the histogram bars
scaled_kde_zinnen = kde_zinnen(x_vals_zinnen) * len(X_zinnen_flat) * np.diff(bins_zinnen)[0]

# Plot KDE for Zinnen as a dashed line
plt.plot(x_vals_zinnen, scaled_kde_zinnen, color='blue', linestyle='--', label='Zinnen (KDE)')

# Repeat the KDE process for Woorden
kde_woorden = gaussian_kde(X_woorden_flat)
x_vals_woorden = np.linspace(X_woorden_flat.min(), X_woorden_flat.max(), 300)
scaled_kde_woorden = kde_woorden(x_vals_woorden) * len(X_woorden_flat) * np.diff(bins_woorden)[0]

# Plot KDE for Woorden as a dashed line
plt.plot(x_vals_woorden, scaled_kde_woorden, color='red', linestyle='--', label='Woorden (KDE)')

# Add axis labels and title
plt.xlabel("acw_50 (s) ")              # X-axis: the actual measurement values
plt.ylabel("Number of values")               # Y-axis: frequency (absolute count)
plt.title("Histogram of acw_50")             # Title of the plot

# Add legend and grid for better readability
plt.legend()
plt.grid(True)

# Adjust layout to avoid clipping of labels or title
plt.tight_layout()

# Show the final plot
plt.show()

# General differences between woorden and zinnen conditions

In [8]:


##dependent condition
def paired_statistic(diff, _):
    return np.mean(diff)

# Crear el diccionario con nombres descriptivos
data_dict = {
    "X_zinnen_subj_mean": X_zinnen_subj_mean,
    "X_woorden_subj_mean": X_woorden_subj_mean}



print("Differences in acw_50 between conditions")
x= data_dict[f"X_zinnen_subj_mean"]
##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
y= data_dict[f"X_woorden_subj_mean"]


diff=x-y
# Permutation test
res = permutation_test(
    (diff, np.zeros_like(diff)),
    #
    statistic=paired_statistic,
    vectorized=False,
    n_resamples=10000,
    alternative='greater', 
    random_state=42
)

print(f"Statistic value: {res.statistic}")
print(f"p-value: {res.pvalue}")

Differences in acw_50 between conditions
Statistic value: 0.00016351881765960357
p-value: 0.28717128287171284


In [9]:
# d of Cohen


mean_diff = np.mean(diff)
std_diff = np.std(diff, ddof=1)  # ddof=1 para usar la desviación muestral

cohens_d = mean_diff / std_diff

print(f"Cohen's d (paired): {cohens_d:.3f}")


Cohen's d (paired): 0.160


# Cluster analysis

## Datos a comparar

ahora tengo por cada condición: subject x epoch x channel

#### procedimiento

- promediar a nivel de epoca para tener subject x channel (luego probaré a hacerlo de otra manera sin promediar)

- ejecutar permutation cluster based analysis usando mi acw_zinnen vs mi acw_woorden  que va a estar como subject x channel

    - el cluster solo puede ser espacial, porque mis epocas son discontinuas, creo que esto es adjacency 

#### Resultados

T_obs: La estadística observada en cada punto.
clusters: Los clústeres formados por puntos significativos adyacentes.
cluster_p_values: Los valores p de cada clúster (ya corregidos por comparaciones múltiples).
H0: La distribución nula obtenida por permutaciones.



Supongo que clusters y sus valores de significacioón
Luego no se como se compara entre condiciones, pero de momemento vamos así

## Cluster in differences

In [10]:
##get adjacency value


adjacency_reduced =  pd.read_pickle(channels_structure_path /f"adjacency_reduced_{modality}.pkl")
###print(f"shape adjacency_reduced: {adjacency_reduced.shape}")

In [11]:
len(subjects)

15

In [12]:
##note that X_zinnen and X_woorden are 2D arrays with shape (n_subjects, n_channels)

diff= X_zinnen - X_woorden
print(f"diff.shape is {diff.shape}")



# as I´m going to apply differences to same subjects, i need a DEPENDENT t test kind of thing, 
# so instead of mne.stats.permutation_cluster_test I´m going to use mne.stats.permutation_cluster_1samp_test

# mne.stats.permutation_cluster_1samp_test(X, threshold=None, n_permutations=1024, tail=0, stat_fun=None, adjacency=None, n_jobs=None, seed=None, max_step=1, exclude=None, 
#                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)

# as im going to put zinnen before woorden i need tail=1

#instead of puting X with bot condigions, i will put the difference between them diff=zinnen-woorden
# i modify the value of the tail to be -1, because i want to see if the values of zinnen are lower than the values of woorden
from scipy.stats import t
p_cluster_value = 0.001
t_thresh = t.ppf(1 - p_cluster_value, df=len(subjects) - 1)

t_obs_diff, clusters_diff, clusters_pv_diff, H0=mne.stats.permutation_cluster_1samp_test(diff, threshold=t_thresh, n_permutations=1024, tail=1, 
                                                                          stat_fun=None, adjacency=adjacency_reduced, n_jobs=15, seed=None, max_step=1, exclude=None,
                                                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)


# Obtenemos los clusters como listas de índices
# Filtrar clusters con p < 0.05
#zip links clusters y p-values
significant_clusters_diff = [
    cluster for cluster, p in zip(clusters_diff, clusters_pv_diff) if p < 0.05
]

print(f"{len(significant_clusters_diff)} significative clusters found.")
print(f"acw_50 Clusters significative: {significant_clusters_diff}")

diff.shape is (15, 273)
stat_fun(H1): min=-1.7145437251732714 max=2.4076954994276627
Running initial clustering …
Found 0 clusters
0 significative clusters found.
acw_50 Clusters significative: []


C:\Users\UCM\AppData\Local\Temp\ipykernel_9248\1268927562.py:22: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  t_obs_diff, clusters_diff, clusters_pv_diff, H0=mne.stats.permutation_cluster_1samp_test(diff, threshold=t_thresh, n_permutations=1024, tail=1,


In [13]:
###PLOTSS

mask_diff = np.zeros(t_obs_diff.shape, dtype=bool)

##significant_clusters_diff is a list of tuples of arrays
# cluster[0] is the electrodes of the  cluster
for cluster in significant_clusters_diff:
    mask_diff[cluster[0]] = True 



##in data i use the value of t_obs_diff
#pos takes the information from evoked.info object
#mask takes the mask_diff object
#in t stat i don´t change it becausse the p value will be calculated with permutations, so no problem when using a parametric test,
# even if the distribution is non parametric 

# Crear figura y eje
fig, ax = plt.subplots(figsize=(6, 6))

# Dibujar el topomap
im, _ = mne.viz.plot_topomap(
    data=t_obs_diff,
    pos=info,
    mask=mask_diff,
    axes=ax,
    cmap='RdBu_r',
    vlim=(-np.max(np.abs(t_obs_diff)), np.max(np.abs(t_obs_diff))),
    mask_params=dict(marker='o', markerfacecolor='yellow', markersize=7),
    contours=0,
    show=False
)

# Añadir título
fig.suptitle(f"acw_50 Clusters in differences, p_value={p_cluster_value}", fontsize=14)

# Añadir barra de color
cbar = fig.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label("T-statistic value")

# Mostrar el gráfico
plt.show()

## plot topomap for word condition

In [14]:

# as I´m going to apply woordenerences to same subjects, i need a DEPENDENT t test kind of thing, 
# so instead of mne.stats.permutation_cluster_test I´m going to use mne.stats.permutation_cluster_1samp_test

# mne.stats.permutation_cluster_1samp_test(X, threshold=None, n_permutations=1024, tail=0, stat_fun=None, adjacency=None, n_jobs=None, seed=None, max_step=1, exclude=None, 
#                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)

# as im going to put zinnen before woorden i need tail=1

#instead of puting X with bot condigions, i will put the woordenerence between them woorden=zinnen-woorden

p_cluster_value = 0.001
t_thresh = t.ppf(1 - p_cluster_value, df=len(subjects) - 1)

t_obs_woorden, clusters_woorden, clusters_pv_woorden, H0=mne.stats.permutation_cluster_1samp_test(X_woorden, threshold=t_thresh, n_permutations=1024, tail=1, 
                                                                          stat_fun=None, adjacency=adjacency_reduced, n_jobs=15, seed=None, max_step=1, exclude=None,
                                                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)


# Obtenemos los clusters como listas de índices
# Filtrar clusters con p < 0.05
#zip links clusters y p-values
significant_clusters_woorden = [
    cluster for cluster, p in zip(clusters_woorden, clusters_pv_woorden) if p < 0.05
]

print(f"{len(significant_clusters_woorden)} significative clusters found.")
print(f"Clusters significative: {significant_clusters_woorden}")

mask_woorden = np.zeros(t_obs_woorden.shape, dtype=bool)

##significant_clusters_woorden is a list of tuples of arrays
# cluster[0] is the electrodes of the  cluster
for cluster in significant_clusters_woorden:
    mask_woorden[cluster[0]] = True 



##in data i use the value of t_obs_woorden
#pos takes the information from evoked.info object
#mask takes the mask_woorden object
#in t stat i don´t change it becausse the p value will be calculated with permutations, so no problem when using a parametric test,
# even if the distribution is non parametric 
# Plot del topomap para la condición "woorden"

# Crear figura y eje
fig, ax = plt.subplots(figsize=(6, 6))

# Dibujar el topomap dentro del eje personalizado
im, _ = mne.viz.plot_topomap(
    data=t_obs_woorden,
    pos=info,
    mask=mask_woorden,
    axes=ax,
    cmap='RdBu_r',
    vlim=(-np.max(np.abs(t_obs_woorden)), np.max(np.abs(t_obs_woorden))),
    mask_params=dict(marker='o', markerfacecolor='yellow', markersize=7),
    contours=0,
    show=False
)

# Añadir título y colorbar
fig.suptitle(f"acw_50 Clusters in Woorden, p_value={p_cluster_value}", fontsize=14)
cbar = fig.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label("T-statistic value")

# Mostrar el gráfico
plt.show()

stat_fun(H1): min=3.363052500728518 max=37.707848526858506
Running initial clustering …
Found 1 cluster


c:\Users\UCM\anaconda3\envs\env_meg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| Permuting : 1023/1023 [00:03<00:00,  316.27it/s]


1 significative clusters found.
Clusters significative: [(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
      

## permutation cluster for zinnen

In [15]:


# as I´m going to apply zinnenerences to same subjects, i need a DEPENDENT t test kind of thing, 
# so instead of mne.stats.permutation_cluster_test I´m going to use mne.stats.permutation_cluster_1samp_test

# mne.stats.permutation_cluster_1samp_test(X, threshold=None, n_permutations=1024, tail=0, stat_fun=None, adjacency=None, n_jobs=None, seed=None, max_step=1, exclude=None, 
#                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)

# as im going to put zinnen before zinnen i need tail=1

#instead of puting X with bot condigions, i will put the zinnenerence between them zinnen=zinnen-zinnen

p_cluster_value = 0.001
t_thresh = t.ppf(1 - p_cluster_value, df=len(subjects) - 1)

t_obs_zinnen, clusters_zinnen, clusters_pv_zinnen, H0=mne.stats.permutation_cluster_1samp_test(X_zinnen, threshold=t_thresh, n_permutations=1024, tail=1, 
                                                                          stat_fun=None, adjacency=adjacency_reduced, n_jobs=15, seed=None, max_step=1, exclude=None,
                                                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)


# Obtenemos los clusters como listas de índices
# Filtrar clusters con p < 0.05
#zip links clusters y p-values
significant_clusters_zinnen = [
    cluster for cluster, p in zip(clusters_zinnen, clusters_pv_zinnen) if p < 0.05
]

print(f"{len(significant_clusters_zinnen)} significative clusters found.")
print(f"Clusters significative: {significant_clusters_zinnen}")


mask_zinnen = np.zeros(t_obs_zinnen.shape, dtype=bool)

##significant_clusters_zinnen is a list of tuples of arrays
# cluster[0] is the electrodes of the  cluster
for cluster in significant_clusters_zinnen:
    mask_zinnen[cluster[0]] = True 



##in data i use the value of t_obs_zinnen
#pos takes the information from evoked.info object
#mask takes the mask_zinnen object
#in t stat i don´t change it becausse the p value will be calculated with permutations, so no problem when using a parametric test,
# even if the distribution is non parametric 
# Crear figura y eje
fig, ax = plt.subplots(figsize=(6, 6))

# Dibujar topomap con eje personalizado
im, _ = mne.viz.plot_topomap(
    data=t_obs_zinnen,
    pos=info,
    mask=mask_zinnen,
    cmap='RdBu_r',
    vlim=(-np.max(np.abs(t_obs_zinnen)), np.max(np.abs(t_obs_zinnen))),
    mask_params=dict(marker='o', markerfacecolor='yellow', markersize=7),
    contours=0,
    axes=ax,       # ← usar el eje personalizado
    show=False     # ← esperar a mostrar
)

# Título
fig.suptitle("acw_50 clusters in zinnen, p_cluster_value={p_cluster_value}", fontsize=14)

# Añadir barra de color correctamente
cbar = fig.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label("T-statistic value")


stat_fun(H1): min=3.4530202663198595 max=35.47761979799943
Running initial clustering …
Found 1 cluster


100%|██████████| Permuting : 1023/1023 [00:00<00:00, 12097.66it/s]

1 significative clusters found.
Clusters significative: [(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
      